## Using the Code Execution Tool

### Installing Libraries

In [ ]:
%pip install anthropic python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
claude_model = os.getenv("CLAUDE_MODEL")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=anthropic_api_key)

### Calling the Messages API with Code Execution Tool

In [ ]:
response = client.messages.create(
    model=claude_model,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Use the code execution tool to calculate the mean and standard deviation of [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",
        }
    ],
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
)

for block in response.content:
    if block.type == "text":
        print(block.text)

### Uploading File for Analysis

In [ ]:
from pathlib import Path

file_object = client.beta.files.upload(
    file=Path("./Docs/indian_cricket_team_score.csv"))

### Calling the Messages API with Code Execution Tool for Analysis

In [ ]:
user_prompt = """
Analyze the attached indian_cricket_team_score.csv dataset using Python.

Generate a Matplotlib bar chart showing the total run scored by each player.

Then:

- Identify the highest-performing player.
- Calculate the total boundaries scored by each player.
- Summarize your findings in a few bullet points.

Use Python code execution for all calculations and visualizations.
"""

# Use the file_id with code execution
response = client.beta.messages.create(
    model=claude_model,
    betas=["files-api-2025-04-14"],
    max_tokens=4096,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_prompt},
                {"type": "container_upload", "file_id": file_object.id},
            ],
        }
    ],
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
)

# display the text-based assistant response
for block in response.content:
    if block.type == "text":
        print(block.text)

### Retrieving Generated File

In [ ]:
from anthropic.types.beta import BetaMessage

# Extract file IDs from the response


def extract_file_ids(response: BetaMessage) -> list[str]:
    file_ids: list[str] = []
    for item in response.content:
        if item.type == "bash_code_execution_tool_result":
            content_item = item.content
            if content_item.type == "bash_code_execution_result":
                for output_block in content_item.content:
                    file_ids.append(output_block.file_id)
    return file_ids


# Download the created files
for file_id in extract_file_ids(response):
    file_metadata = client.beta.files.retrieve_metadata(file_id)
    file_content = client.beta.files.download(file_id)
    file_content.write_to_file(file_metadata.filename)
    print(f"Downloaded: {file_metadata.filename}")